<a href="https://colab.research.google.com/github/JordanTerwilliger/Intro-to-Deep-Learning/blob/main/HW4/HW4_NLP_Transformer_Q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Jordan Terwilliger, 801343938, HW4

In [109]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import requests

from torch import nn
from torch import functional as F
from torch import optim

!pip install torchinfo

import matplotlib.pyplot as plt

from torchinfo import summary

KeyboardInterrupt: 

In [ ]:
torch.manual_seed(1)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text  # This is the entire text data
chars = sorted(list(set(text)))
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}
# Encode the text into integers
all_data = torch.tensor([char_to_int[ch] for ch in text], dtype=torch.long)

# 2. Split into Train and Test slices directly (80% / 20%)
n = int(0.8 * len(all_data))
train_data = all_data[:n]
val_data = all_data[n:]

# 3. Fast random batch fetcher function (No DataLoader overhead!)
def get_batch(split, batch_size=4096, sequence_length=128, device='cuda'):
    data = train_data if split == 'train' else val_data

    # Generate random starting indices for the batch
    ix = torch.randint(len(data) - sequence_length, (batch_size,))

    # Extract sequences and target characters using tensor slicing
    x = torch.stack([data[i : i + sequence_length] for i in ix])
    y = torch.stack([data[i + sequence_length] for i in ix]) # Shape becomes [4096]

    # Push immediately to device
    return x.to(device), y.to(device)

In [ ]:
# We need to convert this text into a list of sorted indices for
print(f"Total Input Characters: {len(text)}")
sorted_text = list(sorted(set(text)))
print(f"Total Unique Characters: {len(sorted_text)}")
print(sorted_text)

ix_to_char = {i: ch for i,ch in enumerate(sorted_text)}
print(ix_to_char)

char_to_ix = {ch: i for i, ch in enumerate(sorted_text)}

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.encoding = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        self.encoding[:, 0::2] = torch.sin(position * div_term)
        self.encoding[:, 1::2] = torch.cos(position * div_term)
        self.encoding = self.encoding.unsqueeze(0).to(device)

    def forward(self, x):
        return x + self.encoding[:, :x.size(1)].detach()

#Create Model
class TransformerModel(nn.Module):
  def __init__(self, input_size, output_size, hidden_size, nhead, num_layers):
    super(TransformerModel, self).__init__()
    self.embedding = nn.Embedding(input_size, hidden_size)
    self.pos_encoder = PositionalEncoding(hidden_size)
    encoder_layers = nn.TransformerEncoderLayer(hidden_size, nhead, batch_first=True)
    self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
    self.fc = nn.Linear(hidden_size, output_size)

  def forward(self, x):
    embedded = self.pos_encoder(self.embedding(x))
    transformer_output = self.transformer_encoder(embedded)
    return self.fc(transformer_output[:, -1, :])

In [ ]:
def createPlot(sequence_length, train_loss_list, val_loss_list, val_accuracy_list):
  plt.figure(figsize=(12, 5))

  # Plot Loss
  plt.subplot(1, 2, 1)
  plt.plot(train_loss_list, label='Train Loss')
  plt.plot(val_loss_list, label='Val Loss')
  plt.title(f'Loss Curves Sequence Length:{sequence_length}')
  plt.xlabel("Epoch")
  plt.ylabel("Loss")
  plt.legend()

  # Plot Accuracy
  plt.subplot(1, 2, 2)
  plt.plot(val_accuracy_list, label='Val Accuracy')
  plt.title(f'Validation Accuracy Sequence Length:{sequence_length}')
  plt.xlabel('Epoch')
  plt.ylabel('Accuracy (%)')
  plt.legend()

  plt.show()

#2 transformer blocks, 2 heads

In [ ]:
#Hyperparameters
hidden_size = 512
num_layers = 2
nhead = 2
lr = 0.001
epochs = 50
input_size = len(sorted_text)

input_lengths = [20, 30]

model = TransformerModel(len(sorted_text), len(sorted_text), hidden_size, nhead, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = lr)

In [ ]:
for seq_len in input_lengths:


# Training Loop
  #Lists for storing loss and validation values
  train_loss_list = []
  val_loss_list = []
  val_accuracy_list = []
  train_batches = 0

  #Create a new training loop for each input_length
  for epoch in range(epochs):
    X_train, y_train = get_batch('train', sequence_length=seq_len)

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()
    model.train()

    epoch_train_loss = 0.0

    optimizer.zero_grad(set_to_none=True)
    y_pred = model(X_train)  # Forward pass
    loss = criterion(y_pred, y_train)  # Compute loss
    loss.backward()  # Backward pass
    optimizer.step()  # Update model parameters

    epoch_train_loss += loss.item()
    train_batches += 1

    end_event.record()
    torch.cuda.synchronize()
    # Calculate time in milliseconds
    elapsed_time_ms = start_event.elapsed_time(end_event)


# Validation Loop
    model.eval()

    total_val_loss = 0.0
    total_val_acc = 0.0
    val_batches = 0

    with torch.no_grad():
      X_val, y_val = get_batch('test', sequence_length=seq_len)

      val_output = model(X_val)
      val_loss = criterion(val_output, y_val) #Find the loss

      predicted = torch.argmax(val_output, 1) #Here we find what the output was (what letter)
      val_accuracy = (predicted == y_val).float().mean()

      total_val_loss += val_loss.item()
      total_val_acc += val_accuracy.item()
      val_batches += 1

    # Compute genuine averages across the whole dataset split
    avg_train_loss = epoch_train_loss / train_batches
    avg_val_loss = total_val_loss / val_batches
    avg_val_acc = total_val_acc / val_batches

    # Append true averages to history lists
    train_loss_list.append(avg_train_loss)
    val_loss_list.append(avg_val_loss)
    val_accuracy_list.append(avg_val_acc)

    if (epoch % 1 == 0):
      print(f'Epoch {epoch}, Loss: {avg_train_loss:.4f}, Val Accuracy: {avg_val_acc:.4f}, Val Loss: {avg_val_loss:.4f}')
      print(f"Training time for Epoch {epoch}: {elapsed_time_ms / 1000:.6f} seconds")
  createPlot(seq_len, train_loss_list, val_loss_list, val_accuracy_list)
  print(summary(model, input_size = (1,input_size), dtypes=[torch.long]))



In [ ]:
def predictNextChar(model, char_to_ix, ix_to_char, initial_str):
  model.eval()
  with torch.no_grad():
    initial_input = torch.tensor([char_to_ix[c] for c in initial_str], dtype=torch.long).unsqueeze(0).to(device)
    prediction = model(initial_input)
    predicted_index = torch.argmax(prediction, dim=1).item()
    return ix_to_char[predicted_index]

In [ ]:
test_str = " "
for _ in range(30):
  predicted_char = predictNextChar(model, char_to_ix, ix_to_char, test_str)
  test_str += predicted_char
  print(f"Predicted next character: '{test_str}'")

#1 transformer block, 2 heads

In [ ]:
#Hyperparameters
num_layers = 1
nhead = 2

model = TransformerModel(len(sorted_text), len(sorted_text), hidden_size, nhead, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = lr)

In [ ]:
for seq_len in input_lengths:


# Training Loop
  #Lists for storing loss and validation values
  train_loss_list = []
  val_loss_list = []
  val_accuracy_list = []
  train_batches = 0

  #Create a new training loop for each input_length
  for epoch in range(epochs):
    X_train, y_train = get_batch('train', sequence_length=seq_len)

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()
    model.train()

    epoch_train_loss = 0.0

    optimizer.zero_grad()
    y_pred = model(X_train)  # Forward pass
    loss = criterion(y_pred, y_train)  # Compute loss
    loss.backward()  # Backward pass
    optimizer.step()  # Update model parameters

    epoch_train_loss += loss.item()
    train_batches += 1

    end_event.record()
    torch.cuda.synchronize()
    # Calculate time in milliseconds
    elapsed_time_ms = start_event.elapsed_time(end_event)


# Validation Loop
    model.eval()

    total_val_loss = 0.0
    total_val_acc = 0.0
    val_batches = 0

    with torch.no_grad():
      X_val, y_val = get_batch('test', sequence_length=seq_len)

      val_output = model(X_val)
      val_loss = criterion(val_output, y_val) #Find the loss

      predicted = torch.argmax(val_output, 1) #Here we find what the output was (what letter)
      val_accuracy = (predicted == y_val).float().mean()

      total_val_loss += val_loss.item()
      total_val_acc += val_accuracy.item()
      val_batches += 1

    # Compute genuine averages across the whole dataset split
    avg_train_loss = epoch_train_loss / train_batches
    avg_val_loss = total_val_loss / val_batches
    avg_val_acc = total_val_acc / val_batches

    # Append true averages to history lists
    train_loss_list.append(avg_train_loss)
    val_loss_list.append(avg_val_loss)
    val_accuracy_list.append(avg_val_acc)

    if (epoch % 1 == 0):
      print(f'Epoch {epoch}, Loss: {avg_train_loss:.4f}, Val Accuracy: {avg_val_acc:.4f}, Val Loss: {avg_val_loss:.4f}')
      print(f"Training time for Epoch {epoch}: {elapsed_time_ms / 1000:.6f} seconds")
  createPlot(seq_len, train_loss_list, val_loss_list, val_accuracy_list)
  print(summary(model, input_size = (1,input_size), dtypes=[torch.long]))



In [ ]:
def predictNextChar(model, char_to_ix, ix_to_char, initial_str):
  model.eval()
  with torch.no_grad():
    initial_input = torch.tensor([char_to_ix[c] for c in initial_str], dtype=torch.long).unsqueeze(0).to(device)
    prediction = model(initial_input)
    predicted_index = torch.argmax(prediction, dim=1).item()
    return ix_to_char[predicted_index]

In [ ]:
test_str = " "
for _ in range(30):
  predicted_char = predictNextChar(model, char_to_ix, ix_to_char, test_str)
  test_str += predicted_char
  print(f"Predicted next character: '{test_str}'")

#1 transformer block , 4 heads

In [ ]:
#Hyperparameters
num_layers = 1
nhead = 4

model = TransformerModel(len(sorted_text), len(sorted_text), hidden_size, nhead, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = lr)

In [ ]:
for seq_len in input_lengths:


# Training Loop
  #Lists for storing loss and validation values
  train_loss_list = []
  val_loss_list = []
  val_accuracy_list = []
  train_batches = 0

  #Create a new training loop for each input_length
  for epoch in range(epochs):
    X_train, y_train = get_batch('train', sequence_length=seq_len)

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()
    model.train()

    epoch_train_loss = 0.0

    optimizer.zero_grad()
    y_pred = model(X_train)  # Forward pass
    loss = criterion(y_pred, y_train)  # Compute loss
    loss.backward()  # Backward pass
    optimizer.step()  # Update model parameters

    epoch_train_loss += loss.item()
    train_batches += 1

    end_event.record()
    torch.cuda.synchronize()
    # Calculate time in milliseconds
    elapsed_time_ms = start_event.elapsed_time(end_event)


# Validation Loop
    model.eval()

    total_val_loss = 0.0
    total_val_acc = 0.0
    val_batches = 0

    with torch.no_grad():
      X_val, y_val = get_batch('test', sequence_length=seq_len)

      val_output = model(X_val)
      val_loss = criterion(val_output, y_val) #Find the loss

      predicted = torch.argmax(val_output, 1) #Here we find what the output was (what letter)
      val_accuracy = (predicted == y_val).float().mean()

      total_val_loss += val_loss.item()
      total_val_acc += val_accuracy.item()
      val_batches += 1

    # Compute genuine averages across the whole dataset split
    avg_train_loss = epoch_train_loss / train_batches
    avg_val_loss = total_val_loss / val_batches
    avg_val_acc = total_val_acc / val_batches

    # Append true averages to history lists
    train_loss_list.append(avg_train_loss)
    val_loss_list.append(avg_val_loss)
    val_accuracy_list.append(avg_val_acc)

    if (epoch % 1 == 0):
      print(f'Epoch {epoch}, Loss: {avg_train_loss:.4f}, Val Accuracy: {avg_val_acc:.4f}, Val Loss: {avg_val_loss:.4f}')
      print(f"Training time for Epoch {epoch}: {elapsed_time_ms / 1000:.6f} seconds")
  createPlot(seq_len, train_loss_list, val_loss_list, val_accuracy_list)
  print(summary(model, input_size = (1,input_size), dtypes=[torch.long]))



In [ ]:
def predictNextChar(model, char_to_ix, ix_to_char, initial_str):
  model.eval()
  with torch.no_grad():
    initial_input = torch.tensor([char_to_ix[c] for c in initial_str], dtype=torch.long).unsqueeze(0).to(device)
    prediction = model(initial_input)
    predicted_index = torch.argmax(prediction, dim=1).item()
    return ix_to_char[predicted_index]

In [ ]:
test_str = " "
for _ in range(30):
  predicted_char = predictNextChar(model, char_to_ix, ix_to_char, test_str)
  test_str += predicted_char
  print(f"Predicted next character: '{test_str}'")

#2 transformer blocks, 4 heads

In [ ]:
#Hyperparameters
num_layers = 2
nhead = 4

model = TransformerModel(len(sorted_text), len(sorted_text), hidden_size, nhead, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = lr)

In [ ]:
for seq_len in input_lengths:


# Training Loop
  #Lists for storing loss and validation values
  train_loss_list = []
  val_loss_list = []
  val_accuracy_list = []
  train_batches = 0

  #Create a new training loop for each input_length
  for epoch in range(epochs):
    X_train, y_train = get_batch('train', sequence_length=seq_len)

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()
    model.train()

    epoch_train_loss = 0.0

    optimizer.zero_grad()
    y_pred = model(X_train)  # Forward pass
    loss = criterion(y_pred, y_train)  # Compute loss
    loss.backward()  # Backward pass
    optimizer.step()  # Update model parameters

    epoch_train_loss += loss.item()
    train_batches += 1

    end_event.record()
    torch.cuda.synchronize()
    # Calculate time in milliseconds
    elapsed_time_ms = start_event.elapsed_time(end_event)


# Validation Loop
    model.eval()

    total_val_loss = 0.0
    total_val_acc = 0.0
    val_batches = 0

    with torch.no_grad():
      X_val, y_val = get_batch('test', sequence_length=seq_len)

      val_output = model(X_val)
      val_loss = criterion(val_output, y_val) #Find the loss

      predicted = torch.argmax(val_output, 1) #Here we find what the output was (what letter)
      val_accuracy = (predicted == y_val).float().mean()

      total_val_loss += val_loss.item()
      total_val_acc += val_accuracy.item()
      val_batches += 1

    # Compute genuine averages across the whole dataset split
    avg_train_loss = epoch_train_loss / train_batches
    avg_val_loss = total_val_loss / val_batches
    avg_val_acc = total_val_acc / val_batches

    # Append true averages to history lists
    train_loss_list.append(avg_train_loss)
    val_loss_list.append(avg_val_loss)
    val_accuracy_list.append(avg_val_acc)

    if (epoch % 1 == 0):
      print(f'Epoch {epoch}, Loss: {avg_train_loss:.4f}, Val Accuracy: {avg_val_acc:.4f}, Val Loss: {avg_val_loss:.4f}')
      print(f"Training time for Epoch {epoch}: {elapsed_time_ms / 1000:.6f} seconds")
  createPlot(seq_len, train_loss_list, val_loss_list, val_accuracy_list)
  print(summary(model, input_size = (1,input_size), dtypes=[torch.long]))



In [ ]:
def predictNextChar(model, char_to_ix, ix_to_char, initial_str):
  model.eval()
  with torch.no_grad():
    initial_input = torch.tensor([char_to_ix[c] for c in initial_str], dtype=torch.long).unsqueeze(0).to(device)
    prediction = model(initial_input)
    predicted_index = torch.argmax(prediction, dim=1).item()
    return ix_to_char[predicted_index]

In [ ]:
test_str = " "
for _ in range(30):
  predicted_char = predictNextChar(model, char_to_ix, ix_to_char, test_str)
  test_str += predicted_char
  print(f"Predicted next character: '{test_str}'")

# 4 transformer blocks, 2 heads

In [ ]:
#Hyperparameters
num_layers = 4
nhead = 2


model = TransformerModel(len(sorted_text), len(sorted_text), hidden_size, nhead, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = lr)

In [ ]:
for seq_len in input_lengths:


# Training Loop
  #Lists for storing loss and validation values
  train_loss_list = []
  val_loss_list = []
  val_accuracy_list = []
  train_batches = 0

  #Create a new training loop for each input_length
  for epoch in range(epochs):
    X_train, y_train = get_batch('train', sequence_length=seq_len)

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()
    model.train()

    epoch_train_loss = 0.0

    optimizer.zero_grad()
    y_pred = model(X_train)  # Forward pass
    loss = criterion(y_pred, y_train)  # Compute loss
    loss.backward()  # Backward pass
    optimizer.step()  # Update model parameters

    epoch_train_loss += loss.item()
    train_batches += 1

    end_event.record()
    torch.cuda.synchronize()
    # Calculate time in milliseconds
    elapsed_time_ms = start_event.elapsed_time(end_event)


# Validation Loop
    model.eval()

    total_val_loss = 0.0
    total_val_acc = 0.0
    val_batches = 0

    with torch.no_grad():
      X_val, y_val = get_batch('test', sequence_length=seq_len)

      val_output = model(X_val)
      val_loss = criterion(val_output, y_val) #Find the loss

      predicted = torch.argmax(val_output, 1) #Here we find what the output was (what letter)
      val_accuracy = (predicted == y_val).float().mean()

      total_val_loss += val_loss.item()
      total_val_acc += val_accuracy.item()
      val_batches += 1

    # Compute genuine averages across the whole dataset split
    avg_train_loss = epoch_train_loss / train_batches
    avg_val_loss = total_val_loss / val_batches
    avg_val_acc = total_val_acc / val_batches

    # Append true averages to history lists
    train_loss_list.append(avg_train_loss)
    val_loss_list.append(avg_val_loss)
    val_accuracy_list.append(avg_val_acc)

    if (epoch % 1 == 0):
      print(f'Epoch {epoch}, Loss: {avg_train_loss:.4f}, Val Accuracy: {avg_val_acc:.4f}, Val Loss: {avg_val_loss:.4f}')
      print(f"Training time for Epoch {epoch}: {elapsed_time_ms / 1000:.6f} seconds")
  createPlot(seq_len, train_loss_list, val_loss_list, val_accuracy_list)
  print(summary(model, input_size = (1,input_size), dtypes=[torch.long]))



In [ ]:
def predictNextChar(model, char_to_ix, ix_to_char, initial_str):
  model.eval()
  with torch.no_grad():
    initial_input = torch.tensor([char_to_ix[c] for c in initial_str], dtype=torch.long).unsqueeze(0).to(device)
    prediction = model(initial_input)
    predicted_index = torch.argmax(prediction, dim=1).item()
    return ix_to_char[predicted_index]

In [ ]:
test_str = " "
for _ in range(30):
  predicted_char = predictNextChar(model, char_to_ix, ix_to_char, test_str)
  test_str += predicted_char
  print(f"Predicted next character: '{test_str}'")

# 4 transformer blocks, 4 heads

In [ ]:
#Hyperparameters
num_layers = 4
nhead = 4


model = TransformerModel(len(sorted_text), len(sorted_text), hidden_size, nhead, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = lr)

In [ ]:
for seq_len in input_lengths:
# Training Loop
  #Lists for storing loss and validation values
  train_loss_list = []
  val_loss_list = []
  val_accuracy_list = []
  train_batches = 0

  #Create a new training loop for each input_length
  for epoch in range(epochs):
    X_train, y_train = get_batch('train', sequence_length=seq_len)

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()
    model.train()

    epoch_train_loss = 0.0

    optimizer.zero_grad()
    y_pred = model(X_train)  # Forward pass
    loss = criterion(y_pred, y_train)  # Compute loss
    loss.backward()  # Backward pass
    optimizer.step()  # Update model parameters

    epoch_train_loss += loss.item()
    train_batches += 1

    end_event.record()
    torch.cuda.synchronize()
    # Calculate time in milliseconds
    elapsed_time_ms = start_event.elapsed_time(end_event)


# Validation Loop
    model.eval()

    total_val_loss = 0.0
    total_val_acc = 0.0
    val_batches = 0

    with torch.no_grad():
      X_val, y_val = get_batch('test', sequence_length=seq_len)

      val_output = model(X_val)
      val_loss = criterion(val_output, y_val) #Find the loss

      predicted = torch.argmax(val_output, 1) #Here we find what the output was (what letter)
      val_accuracy = (predicted == y_val).float().mean()

      total_val_loss += val_loss.item()
      total_val_acc += val_accuracy.item()
      val_batches += 1

    # Compute genuine averages across the whole dataset split
    avg_train_loss = epoch_train_loss / train_batches
    avg_val_loss = total_val_loss / val_batches
    avg_val_acc = total_val_acc / val_batches

    # Append true averages to history lists
    train_loss_list.append(avg_train_loss)
    val_loss_list.append(avg_val_loss)
    val_accuracy_list.append(avg_val_acc)

    if (epoch % 1 == 0):
      print(f'Epoch {epoch}, Loss: {avg_train_loss:.4f}, Val Accuracy: {avg_val_acc:.4f}, Val Loss: {avg_val_loss:.4f}')
      print(f"Training time for Epoch {epoch}: {elapsed_time_ms / 1000:.6f} seconds")
  createPlot(seq_len, train_loss_list, val_loss_list, val_accuracy_list)
  print(summary(model, input_size = (1,input_size), dtypes=[torch.long]))



In [ ]:
def predictNextChar(model, char_to_ix, ix_to_char, initial_str):
  model.eval()
  with torch.no_grad():
    initial_input = torch.tensor([char_to_ix[c] for c in initial_str], dtype=torch.long).unsqueeze(0).to(device)
    prediction = model(initial_input)
    predicted_index = torch.argmax(prediction, dim=1).item()
    return ix_to_char[predicted_index]

In [ ]:
test_str = " "
for _ in range(30):
  predicted_char = predictNextChar(model, char_to_ix, ix_to_char, test_str)
  test_str += predicted_char
  print(f"Predicted next character: '{test_str}'")